# Active learning: uma rodada por incerteza

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
INITIAL_SIZE = 20
QUERY_SIZE = 10

## Dados e teste fixo

In [2]:
X, y = make_classification(
    n_samples=600,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    n_classes=2,
    flip_y=0.05,
    random_state=SEED,
)
X_available, X_test, y_available, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED,
)

# A estratificação inicial garante exemplos das duas classes na base rotulada.
X_labeled, X_pool, y_labeled, y_pool_oracle = train_test_split(
    X_available, y_available,
    train_size=INITIAL_SIZE,
    stratify=y_available,
    random_state=SEED,
)
assert set(y_labeled) == {0, 1}
print(f"Rotuladas: {len(y_labeled)} | Pool: {len(X_pool)} | Teste: {len(X_test)}")

Rotuladas: 20 | Pool: 430 | Teste: 150


## Treino inicial

In [3]:
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(solver="liblinear", max_iter=1000, random_state=SEED),
)
model.fit(X_labeled, y_labeled)
accuracy_before = accuracy_score(y_test, model.predict(X_test))
labeled_before = len(y_labeled)
pool_before = len(X_pool)
print(f"Antes: {labeled_before} rotuladas | Acurácia no teste: {accuracy_before:.3f}")

Antes: 20 rotuladas | Acurácia no teste: 0.647


## Seleção por incerteza e anotação simulada

In [4]:
pool_probabilities = model.predict_proba(X_pool)[:, 1]
uncertainty_distance = np.abs(pool_probabilities - 0.5)
query_indices = np.argsort(uncertainty_distance, kind="stable")[:QUERY_SIZE]

print("Probabilidades das amostras selecionadas:")
print(np.round(pool_probabilities[query_indices], 3))

# A consulta ao oráculo ocorre apenas após a seleção e só para essas amostras.
X_queried = X_pool[query_indices]
y_queried = y_pool_oracle[query_indices]

Probabilidades das amostras selecionadas:
[0.5   0.5   0.501 0.502 0.502 0.497 0.504 0.504 0.495 0.505]


## Atualização e novo treino

In [5]:
X_labeled = np.concatenate([X_labeled, X_queried])
y_labeled = np.concatenate([y_labeled, y_queried])

keep_in_pool = np.ones(len(X_pool), dtype=bool)
keep_in_pool[query_indices] = False
X_pool = X_pool[keep_in_pool]
y_pool_oracle = y_pool_oracle[keep_in_pool]

assert len(y_labeled) == labeled_before + QUERY_SIZE
assert len(X_pool) == pool_before - QUERY_SIZE

model.fit(X_labeled, y_labeled)
accuracy_after = accuracy_score(y_test, model.predict(X_test))

print(f"Teste fixo: {len(X_test)} amostras")
print(f"{'Etapa':<10} {'Rotuladas':>10} {'Pool':>8} {'Acurácia':>10}")
print(f"{'Antes':<10} {labeled_before:>10} {pool_before:>8} {accuracy_before:>10.3f}")
print(f"{'Depois':<10} {len(y_labeled):>10} {len(X_pool):>8} {accuracy_after:>10.3f}")

Teste fixo: 150 amostras
Etapa       Rotuladas     Pool   Acurácia
Antes              20      430      0.647
Depois             30      420      0.687
